In [5]:
import hdc
hdc.connect()

Database is already connected.


In [6]:
prov_c = '49'
id_val = 'your_id' # ใส่ ID ที่ต้องการ
cat_id = 'your_cat_id'
# กำหนดวันที่ (ใช้รูปแบบ YYYYMMDD หรือ YYYY-MM-DD ตามที่ฐานข้อมูลเก็บ)
start_d1 = '2024-10-01'
end_d1 = '2025-09-30'
start_d2 = '2025-10-01'
end_d2 = '2026-09-30'

In [10]:
%%sql
/*------------ s_op_instype_all (Telemed Style) ------------*/
DROP TABLE IF EXISTS s_op_instype_all_compare;

CREATE TABLE s_op_instype_all_compare AS
SELECT 
    x.hospcode, 
    c.hosname AS hospname,
    c.amp_code,
    -- ปี 2568
    x.inscl_all_68, x.visit_all_68,
    x.inscl1_68, x.inscl2_68, x.inscl3_68, x.inscl4_68, x.inscl5_68,
    -- ปี 2569
    x.inscl_all_69, x.visit_all_69,
    x.inscl1_69, x.inscl2_69, x.inscl3_69, x.inscl4_69, x.inscl5_69
FROM (
    SELECT s.hospcode
        -- ทั้งหมด ปี 68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}', CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl_all_68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}', CONCAT(s.hospcode,'-',s.pid,'-',s.seq), NULL)) AS visit_all_68
        
        -- ทั้งหมด ปี 69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}', CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl_all_69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}', CONCAT(s.hospcode,'-',s.pid,'-',s.seq), NULL)) AS visit_all_69

        -- แยกกลุ่มสิทธิ ปี 68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('1')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl1_68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('2')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl2_68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('3','4')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl3_68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('5')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl4_68
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d1}}' AND s.instype NOT IN (SELECT instypecode FROM cinstype_new), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl5_68

        -- แยกกลุ่มสิทธิ ปี 69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('1')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl1_69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('2')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl2_69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('3','4')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl3_69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}' AND s.instype IN (SELECT instypecode FROM cinstype_new WHERE instypegroup IN ('5')), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl4_69
        ,COUNT(DISTINCT IF(s.date_serv BETWEEN '{{start_d2}}' AND '{{end_d2}}' AND s.instype NOT IN (SELECT instypecode FROM cinstype_new), CONCAT(s.hospcode,'-',s.pid), NULL)) AS inscl5_69

    FROM service s
    INNER JOIN chospital h ON s.hospcode = h.hoscode
    WHERE s.date_serv BETWEEN '{{start_d1}}' AND '{{end_d2}}'
      AND h.provcode = '{{prov_c}}'
    GROUP BY s.hospcode
) AS x
INNER JOIN chospital c ON c.hoscode = x.hospcode
ORDER BY c.amp_code, c.hoscode;

Count
86


In [11]:
from datetime import datetime
# ชื่อจาก วันที่
today = datetime.today().strftime('%Y%m%d')
filename = f'./Export/{today}_{prov_c}_s_op_instype_all_compare.xlsx'

In [12]:
%%sql
/* Export to excel แบบนี้ work สุด */
install excel;
load excel;
COPY (SELECT * FROM  s_op_instype_all_compare ORDER BY amp_code,hospcode)
TO '{{filename}}' (FORMAT xlsx, HEADER true);

Count
86


In [ ]:
hdc.close()